In [2]:
import torch
import numpy as np
import gc
from sklearn.metrics import confusion_matrix

from model import build_model

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
TEST_PATH = "/home/dani/Documents/tugas akhir/TugasAkhir/codeTugasAkhirku2026/tow_ids/Preprocessing/Preprocessingbaru/imgsize256lv2/norm/dwt/tow_ids_test_dwt.npz"

data = np.load(TEST_PATH)

x_test = data["X"]
y_test = data["y"]

print("Shape:", x_test.shape, y_test.shape)

# numpy → torch
x_test = torch.tensor(x_test, dtype=torch.float32).permute(0, 3, 1, 2)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

Shape: (3092, 64, 64, 3) (3092,)


In [5]:
def evaluate_model(model, x, y, batch_size=16):

    model.eval()

    outputs_list = []

    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            x_batch = x[i:i+batch_size].to(device)
            outputs = model(x_batch)
            outputs_list.append(outputs.cpu())

    outputs = torch.cat(outputs_list, dim=0)

    probs = torch.sigmoid(outputs)
    preds = (probs > 0.5).float()

    y_true = y.numpy().reshape(-1)
    y_pred = preds.numpy().reshape(-1)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print("\n===== HASIL EVALUASI MODEL =====")
    print(f"Accuracy  : {accuracy:.6f}")
    print(f"Precision : {precision:.6f}")
    print(f"Recall    : {recall:.6f}")
    print(f"F1-Score  : {f1:.6f}")
    print(f"FPR       : {fpr:.6f}")
    print(f"FNR       : {fnr:.6f}")

    print("\nConfusion Matrix:")
    print(f"TN: {tn} | FP: {fp}")
    print(f"FN: {fn} | TP: {tp}")

In [6]:
FINAL_MODEL_PATH = "saved_models/model_final_iid.pt"

model = build_model(device)

state_dict = torch.load(FINAL_MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)

print("\n[TEST] FINAL MODEL")

evaluate_model(model, x_test, y_test)

/tmp/ipykernel_35456/2117306466.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(FINAL_MODEL_PATH, map_location=device)



[TEST] FINAL MODEL

===== HASIL EVALUASI MODEL =====
Accuracy  : 0.957309
Precision : 0.967675
Recall    : 0.943029
F1-Score  : 0.955193
FPR       : 0.029375
FNR       : 0.056971

Confusion Matrix:
TN: 1553 | FP: 47
FN: 85 | TP: 1407


In [7]:
del model
torch.cuda.empty_cache()
gc.collect()

print("[INFO] Memory dibersihkan")

[INFO] Memory dibersihkan


In [8]:
BEST_MODEL_PATH = "saved_models/model_best_iid.pt"

model = build_model(device)

state_dict = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)

print("\n[TEST] BEST MODEL")

evaluate_model(model, x_test, y_test)

/tmp/ipykernel_35456/1449204542.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(BEST_MODEL_PATH, map_location=device)



[TEST] BEST MODEL

===== HASIL EVALUASI MODEL =====
Accuracy  : 0.962807
Precision : 0.963636
Recall    : 0.959115
F1-Score  : 0.961371
FPR       : 0.033750
FNR       : 0.040885

Confusion Matrix:
TN: 1546 | FP: 54
FN: 61 | TP: 1431
